# AILS Large-Scale Grid Experiments

Evaluate AILS algorithm behavior on large grid maps (1000x1000, 5000x5000, 10000x10000)
to assess scalability at scales significantly beyond the standard experimental range.

**Author:** Amr Elshahed  
**Institution:** Universiti Sains Malaysia

---

## Experiment Overview

| Grid Size | Cells | Trials | Timeout/trial | Algorithms |
|-----------|-------|--------|---------------|------------|
| 1000x1000 | 1M | 30 | 60s | AILS, A*, JPS |
| 5000x5000 | 25M | 15 | 180s | AILS, A*, JPS |
| 10000x10000 | 100M | 10 | 300s | AILS, A*, JPS |

**Key Questions:**
1. How does AILS execution time scale at 1K, 5K, 10K grid sizes?
2. How does corridor efficiency change at large scales?
3. What is the memory footprint at large scales?
4. How does the AILS advantage over A*/JPS change with scale?
5. Does the adaptive corridor strategy maintain its advantage at scale?

In [ ]:
# Setup: imports and path configuration
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import time
from pathlib import Path
from datetime import datetime

# Add experiment root to path
EXPERIMENT_ROOT = str(Path(os.getcwd()).parent)
if EXPERIMENT_ROOT not in sys.path:
    sys.path.insert(0, EXPERIMENT_ROOT)

from large_scale_grids.large_scale_benchmark import (
    LargeScaleBenchmark,
    LargeScaleConfig,
    LargeScaleResult,
)

# Plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 11

# Results directory
RESULTS_DIR = 'results/large_scale_grids'
os.makedirs(RESULTS_DIR, exist_ok=True)

print('Setup complete!')
print(f'Experiment root: {EXPERIMENT_ROOT}')

## 1. Configuration

Choose your experiment configuration below. Three options:
- **Quick**: 1000x1000 only, for testing the pipeline
- **Medium**: 1000x1000 and 5000x5000
- **Full**: All three scales (1000, 5000, 10000)

In [ ]:
# ============================================================
# SELECT EXPERIMENT MODE: 'quick', 'medium', or 'full'
# ============================================================
EXPERIMENT_MODE = 'full'  # <-- Change this to control scope
SEED = 42
# ============================================================

if EXPERIMENT_MODE == 'quick':
    config = LargeScaleConfig(
        grid_sizes=[(1000, 1000)],
        densities=[0.2],
        patterns=['random'],
        algorithms=['AILS', 'A*', 'JPS'],
        trials_by_size={1000: 10},
        timeout_per_trial={1000: 60},
        seed=SEED,
        output_dir=RESULTS_DIR,
    )
elif EXPERIMENT_MODE == 'medium':
    config = LargeScaleConfig(
        grid_sizes=[(1000, 1000), (5000, 5000)],
        densities=[0.2, 0.3],
        patterns=['random', 'clustered', 'mixed'],
        algorithms=['AILS', 'A*', 'JPS'],
        trials_by_size={1000: 20, 5000: 10},
        timeout_per_trial={1000: 60, 5000: 180},
        seed=SEED,
        output_dir=RESULTS_DIR,
    )
else:  # full
    config = LargeScaleConfig(
        grid_sizes=[(1000, 1000), (5000, 5000), (10000, 10000)],
        densities=[0.2, 0.3],
        patterns=['random', 'clustered', 'mixed'],
        algorithms=['AILS', 'A*', 'JPS'],
        trials_by_size={1000: 30, 5000: 15, 10000: 10},
        timeout_per_trial={1000: 60, 5000: 180, 10000: 300},
        seed=SEED,
        output_dir=RESULTS_DIR,
    )

print(f'Mode: {EXPERIMENT_MODE}')
print(f'Grid sizes: {[f"{w}x{h}" for w, h in config.grid_sizes]}')
print(f'Densities: {config.densities}')
print(f'Patterns: {config.patterns}')
print(f'Algorithms: {config.algorithms}')
print(f'Trials: {config.trials_by_size}')
print(f'Timeouts: {config.timeout_per_trial}')

## 2. Run Experiments

This cell runs all experiments. Depending on the mode:
- **Quick**: ~5 minutes
- **Medium**: ~30-60 minutes  
- **Full**: several hours (depends on hardware)

In [ ]:
benchmark = LargeScaleBenchmark(config)
results = benchmark.run(verbose=True)

In [ ]:
# Save raw results
csv_path, json_path = benchmark.save_results()
print(f'CSV saved: {csv_path}')
print(f'JSON saved: {json_path}')

## 3. Load Results into DataFrame

Convert results to a pandas DataFrame for analysis.

In [ ]:
# Convert to DataFrame
df = pd.DataFrame([r.to_dict() for r in results])

# Add derived columns
df['grid_size_label'] = df['grid_width'].astype(str) + 'x' + df['grid_height'].astype(str)
df['successful'] = (~df['timed_out']) & (df['path_found'])

print(f'Total rows: {len(df)}')
print(f'Successful: {df["successful"].sum()}')
print(f'Timed out: {df["timed_out"].sum()}')
print()
print(df.groupby(['grid_size_label', 'algorithm'])['successful'].mean().unstack())

In [ ]:
# Filter to successful trials for analysis
df_ok = df[df['successful']].copy()
print(f'Successful trials for analysis: {len(df_ok)}')
df_ok.head()

## 4. Performance Analysis by Grid Size

Core results: execution time, nodes visited, memory across scales.

In [ ]:
# Summary table: mean performance by grid size and algorithm
summary = df_ok.groupby(['grid_size_label', 'algorithm']).agg(
    time_mean=('execution_time_ms', 'mean'),
    time_std=('execution_time_ms', 'std'),
    time_median=('execution_time_ms', 'median'),
    nodes_mean=('visited_nodes', 'mean'),
    nodes_std=('visited_nodes', 'std'),
    memory_mean=('memory_peak_mb', 'mean'),
    memory_max=('memory_peak_mb', 'max'),
    trials=('execution_time_ms', 'count'),
).round(2)

print('=== Performance Summary ===')
print(summary.to_string())

# Save for reference
summary.to_csv(f'{RESULTS_DIR}/performance_summary.csv')

In [ ]:
# Visualization: Execution Time by Grid Size
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

grid_order = sorted(df_ok['grid_size_label'].unique(),
                    key=lambda x: int(x.split('x')[0]))

# 1. Execution Time
ax = axes[0]
for algo in df_ok['algorithm'].unique():
    data = df_ok[df_ok['algorithm'] == algo]
    means = data.groupby('grid_size_label')['execution_time_ms'].mean()
    stds = data.groupby('grid_size_label')['execution_time_ms'].std()
    means = means.reindex(grid_order)
    stds = stds.reindex(grid_order)
    ax.errorbar(range(len(grid_order)), means.values, yerr=stds.values,
                marker='o', capsize=5, label=algo, linewidth=2)
ax.set_xticks(range(len(grid_order)))
ax.set_xticklabels(grid_order)
ax.set_xlabel('Grid Size')
ax.set_ylabel('Execution Time (ms)')
ax.set_title('Execution Time vs Grid Size')
ax.legend()
ax.set_yscale('log')

# 2. Nodes Visited
ax = axes[1]
for algo in df_ok['algorithm'].unique():
    data = df_ok[df_ok['algorithm'] == algo]
    means = data.groupby('grid_size_label')['visited_nodes'].mean()
    stds = data.groupby('grid_size_label')['visited_nodes'].std()
    means = means.reindex(grid_order)
    stds = stds.reindex(grid_order)
    ax.errorbar(range(len(grid_order)), means.values, yerr=stds.values,
                marker='o', capsize=5, label=algo, linewidth=2)
ax.set_xticks(range(len(grid_order)))
ax.set_xticklabels(grid_order)
ax.set_xlabel('Grid Size')
ax.set_ylabel('Nodes Visited')
ax.set_title('Nodes Visited vs Grid Size')
ax.legend()
ax.set_yscale('log')

# 3. Memory Usage
ax = axes[2]
for algo in df_ok['algorithm'].unique():
    data = df_ok[df_ok['algorithm'] == algo]
    means = data.groupby('grid_size_label')['memory_peak_mb'].mean()
    means = means.reindex(grid_order)
    ax.plot(range(len(grid_order)), means.values,
            marker='o', label=algo, linewidth=2)
ax.set_xticks(range(len(grid_order)))
ax.set_xticklabels(grid_order)
ax.set_xlabel('Grid Size')
ax.set_ylabel('Peak Memory (MB)')
ax.set_title('Memory Usage vs Grid Size')
ax.legend()

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/performance_by_grid_size.png', bbox_inches='tight')
plt.show()

## 5. Speedup Analysis: AILS vs A* and JPS

In [ ]:
# Compute speedup of AILS over A* and JPS at each grid size
speedup_data = []

for size in grid_order:
    size_data = df_ok[df_ok['grid_size_label'] == size]
    ails_time = size_data[size_data['algorithm'] == 'AILS']['execution_time_ms'].mean()
    ails_nodes = size_data[size_data['algorithm'] == 'AILS']['visited_nodes'].mean()

    for baseline in ['A*', 'JPS']:
        bl_data = size_data[size_data['algorithm'] == baseline]
        if len(bl_data) == 0:
            continue
        bl_time = bl_data['execution_time_ms'].mean()
        bl_nodes = bl_data['visited_nodes'].mean()

        speedup_data.append({
            'grid_size': size,
            'baseline': baseline,
            'time_speedup': bl_time / ails_time if ails_time > 0 else 0,
            'time_reduction_pct': (1 - ails_time / bl_time) * 100 if bl_time > 0 else 0,
            'node_reduction_pct': (1 - ails_nodes / bl_nodes) * 100 if bl_nodes > 0 else 0,
        })

df_speedup = pd.DataFrame(speedup_data)
print('=== AILS Speedup Over Baselines ===')
print(df_speedup.to_string(index=False))
df_speedup.to_csv(f'{RESULTS_DIR}/speedup_analysis.csv', index=False)

In [ ]:
# Visualization: Speedup trend
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Time speedup
ax = axes[0]
for baseline in df_speedup['baseline'].unique():
    data = df_speedup[df_speedup['baseline'] == baseline]
    ax.plot(range(len(data)), data['time_speedup'].values,
            marker='o', linewidth=2, label=f'AILS vs {baseline}')
ax.set_xticks(range(len(grid_order)))
ax.set_xticklabels(grid_order)
ax.set_xlabel('Grid Size')
ax.set_ylabel('Speedup Factor (x)')
ax.set_title('AILS Time Speedup Over Baselines')
ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
ax.legend()

# Node reduction
ax = axes[1]
for baseline in df_speedup['baseline'].unique():
    data = df_speedup[df_speedup['baseline'] == baseline]
    ax.plot(range(len(data)), data['node_reduction_pct'].values,
            marker='o', linewidth=2, label=f'AILS vs {baseline}')
ax.set_xticks(range(len(grid_order)))
ax.set_xticklabels(grid_order)
ax.set_xlabel('Grid Size')
ax.set_ylabel('Node Reduction (%)')
ax.set_title('AILS Node Reduction Over Baselines')
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.legend()

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/speedup_trend.png', bbox_inches='tight')
plt.show()

## 6. Scaling Analysis (Power Law Fit)

In [ ]:
# Fit power law: time ~ n^b  (n = grid_cells)
scaling_results = []

for algo in df_ok['algorithm'].unique():
    algo_data = df_ok[df_ok['algorithm'] == algo]
    sizes = sorted(algo_data['grid_cells'].unique())
    mean_times = [algo_data[algo_data['grid_cells'] == s]['execution_time_ms'].mean() for s in sizes]
    mean_nodes = [algo_data[algo_data['grid_cells'] == s]['visited_nodes'].mean() for s in sizes]

    if len(sizes) >= 2:
        log_sizes = np.log(sizes)
        log_times = np.log(np.array(mean_times) + 1e-10)
        coeffs = np.polyfit(log_sizes, log_times, 1)
        time_exponent = coeffs[0]

        log_nodes = np.log(np.array(mean_nodes) + 1e-10)
        node_coeffs = np.polyfit(log_sizes, log_nodes, 1)
        node_exponent = node_coeffs[0]
    else:
        time_exponent = 0
        node_exponent = 0

    scaling_results.append({
        'algorithm': algo,
        'time_exponent': time_exponent,
        'node_exponent': node_exponent,
        'sizes': sizes,
        'mean_times': mean_times,
        'mean_nodes': mean_nodes,
    })

df_scaling = pd.DataFrame([{
    'Algorithm': r['algorithm'],
    'Time Exponent (b)': f"{r['time_exponent']:.3f}",
    'Node Exponent': f"{r['node_exponent']:.3f}",
} for r in scaling_results])

print('=== Scaling Analysis (time ~ n^b) ===')
print(df_scaling.to_string(index=False))
df_scaling.to_csv(f'{RESULTS_DIR}/scaling_analysis.csv', index=False)

In [ ]:
# Visualization: log-log scaling plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Time scaling
ax = axes[0]
for r in scaling_results:
    ax.loglog(r['sizes'], r['mean_times'],
              marker='o', linewidth=2, label=f"{r['algorithm']} (b={r['time_exponent']:.2f})")
ax.set_xlabel('Grid Cells (n)')
ax.set_ylabel('Execution Time (ms)')
ax.set_title('Time Scaling (log-log)')
ax.legend()
ax.grid(True, alpha=0.3)

# Node scaling
ax = axes[1]
for r in scaling_results:
    ax.loglog(r['sizes'], r['mean_nodes'],
              marker='o', linewidth=2, label=f"{r['algorithm']} (b={r['node_exponent']:.2f})")
ax.set_xlabel('Grid Cells (n)')
ax.set_ylabel('Nodes Visited')
ax.set_title('Node Scaling (log-log)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/scaling_loglog.png', bbox_inches='tight')
plt.show()

## 7. AILS Corridor Behavior at Scale

In [ ]:
# Analyze AILS corridor metrics at each scale
df_ails = df_ok[df_ok['algorithm'] == 'AILS'].copy()

if len(df_ails) > 0:
    df_ails['corridor_to_grid_ratio'] = df_ails['corridor_size'] / df_ails['grid_cells']
    df_ails['nodes_per_corridor_cell'] = df_ails['visited_nodes'] / df_ails['corridor_size'].clip(lower=1)

    corridor_summary = df_ails.groupby('grid_size_label').agg(
        corridor_efficiency_mean=('corridor_efficiency', 'mean'),
        corridor_efficiency_std=('corridor_efficiency', 'std'),
        corridor_size_mean=('corridor_size', 'mean'),
        corridor_to_grid_ratio=('corridor_to_grid_ratio', 'mean'),
        nodes_per_corridor=('nodes_per_corridor_cell', 'mean'),
    ).round(4)

    print('=== AILS Corridor Behavior at Scale ===')
    print(corridor_summary.to_string())
    corridor_summary.to_csv(f'{RESULTS_DIR}/corridor_analysis.csv')
else:
    print('No AILS results available.')

In [ ]:
# Visualization: Corridor metrics
if len(df_ails) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Corridor Efficiency
    ax = axes[0]
    corridor_summary_sorted = corridor_summary.reindex(
        sorted(corridor_summary.index, key=lambda x: int(x.split('x')[0]))
    )
    ax.bar(range(len(corridor_summary_sorted)),
           corridor_summary_sorted['corridor_efficiency_mean'].values,
           yerr=corridor_summary_sorted['corridor_efficiency_std'].values,
           capsize=5, color='steelblue', alpha=0.8)
    ax.set_xticks(range(len(corridor_summary_sorted)))
    ax.set_xticklabels(corridor_summary_sorted.index)
    ax.set_xlabel('Grid Size')
    ax.set_ylabel('Corridor Efficiency')
    ax.set_title('Corridor Efficiency vs Grid Size')

    # Corridor-to-Grid Ratio
    ax = axes[1]
    ax.bar(range(len(corridor_summary_sorted)),
           corridor_summary_sorted['corridor_to_grid_ratio'].values * 100,
           color='coral', alpha=0.8)
    ax.set_xticks(range(len(corridor_summary_sorted)))
    ax.set_xticklabels(corridor_summary_sorted.index)
    ax.set_xlabel('Grid Size')
    ax.set_ylabel('Corridor / Grid (%)')
    ax.set_title('Search Space Fraction vs Grid Size')

    # Absolute Corridor Size
    ax = axes[2]
    ax.bar(range(len(corridor_summary_sorted)),
           corridor_summary_sorted['corridor_size_mean'].values,
           color='mediumseagreen', alpha=0.8)
    ax.set_xticks(range(len(corridor_summary_sorted)))
    ax.set_xticklabels(corridor_summary_sorted.index)
    ax.set_xlabel('Grid Size')
    ax.set_ylabel('Corridor Size (cells)')
    ax.set_title('Absolute Corridor Size vs Grid Size')

    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/corridor_behavior.png', bbox_inches='tight')
    plt.show()

## 8. Performance by Obstacle Pattern

In [ ]:
# Performance breakdown by pattern
pattern_summary = df_ok.groupby(['pattern', 'algorithm']).agg(
    time_mean=('execution_time_ms', 'mean'),
    time_std=('execution_time_ms', 'std'),
    nodes_mean=('visited_nodes', 'mean'),
    nodes_std=('visited_nodes', 'std'),
    memory_mean=('memory_peak_mb', 'mean'),
    count=('execution_time_ms', 'count'),
).round(2)

print('=== Performance by Obstacle Pattern ===')
print(pattern_summary.to_string())
pattern_summary.to_csv(f'{RESULTS_DIR}/pattern_analysis.csv')

In [ ]:
# Visualization: Pattern comparison
patterns = sorted(df_ok['pattern'].unique())
algos = sorted(df_ok['algorithm'].unique())

if len(patterns) > 1:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Time by pattern
    ax = axes[0]
    x = np.arange(len(patterns))
    bar_width = 0.8 / len(algos)
    for i, algo in enumerate(algos):
        vals = []
        for p in patterns:
            subset = df_ok[(df_ok['pattern'] == p) & (df_ok['algorithm'] == algo)]
            vals.append(subset['execution_time_ms'].mean() if len(subset) > 0 else 0)
        ax.bar(x + i * bar_width, vals, bar_width, label=algo)
    ax.set_xticks(x + bar_width * (len(algos) - 1) / 2)
    ax.set_xticklabels([p.capitalize() for p in patterns])
    ax.set_ylabel('Time (ms)')
    ax.set_title('Execution Time by Obstacle Pattern')
    ax.legend()

    # Nodes by pattern
    ax = axes[1]
    for i, algo in enumerate(algos):
        vals = []
        for p in patterns:
            subset = df_ok[(df_ok['pattern'] == p) & (df_ok['algorithm'] == algo)]
            vals.append(subset['visited_nodes'].mean() if len(subset) > 0 else 0)
        ax.bar(x + i * bar_width, vals, bar_width, label=algo)
    ax.set_xticks(x + bar_width * (len(algos) - 1) / 2)
    ax.set_xticklabels([p.capitalize() for p in patterns])
    ax.set_ylabel('Nodes Visited')
    ax.set_title('Nodes Visited by Obstacle Pattern')
    ax.legend()

    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/pattern_comparison.png', bbox_inches='tight')
    plt.show()
else:
    print('Only one pattern tested, skipping pattern comparison plot.')

## 9. Statistical Significance Testing

In [ ]:
from scipy import stats as scipy_stats

stat_results = []

for size in grid_order:
    size_data = df_ok[df_ok['grid_size_label'] == size]
    ails_times = size_data[size_data['algorithm'] == 'AILS']['execution_time_ms'].values

    for baseline in ['A*', 'JPS']:
        bl_times = size_data[size_data['algorithm'] == baseline]['execution_time_ms'].values

        if len(ails_times) < 3 or len(bl_times) < 3:
            continue

        min_len = min(len(ails_times), len(bl_times))

        # Paired t-test
        t_stat, p_value = scipy_stats.ttest_rel(
            ails_times[:min_len], bl_times[:min_len]
        )

        # Wilcoxon signed-rank test
        try:
            w_stat, w_pvalue = scipy_stats.wilcoxon(
                ails_times[:min_len], bl_times[:min_len]
            )
        except ValueError:
            w_stat, w_pvalue = 0, 1.0

        # Cohen's d
        pooled_std = np.sqrt(
            (np.var(ails_times[:min_len]) + np.var(bl_times[:min_len])) / 2
        )
        cohens_d = (
            (np.mean(bl_times[:min_len]) - np.mean(ails_times[:min_len]))
            / pooled_std
        ) if pooled_std > 0 else 0

        effect_interp = (
            'large' if abs(cohens_d) > 0.8
            else 'medium' if abs(cohens_d) > 0.5
            else 'small'
        )

        stat_results.append({
            'Grid Size': size,
            'Comparison': f'AILS vs {baseline}',
            't-statistic': round(t_stat, 4),
            'p-value (t-test)': round(p_value, 6),
            'Wilcoxon p-value': round(w_pvalue, 6),
            'Cohen\'s d': round(cohens_d, 4),
            'Effect Size': effect_interp,
            'Significant (p<0.05)': p_value < 0.05,
        })

df_stats = pd.DataFrame(stat_results)
print('=== Statistical Significance Tests ===')
print(df_stats.to_string(index=False))
df_stats.to_csv(f'{RESULTS_DIR}/statistical_tests.csv', index=False)

## 10. Normalized Metrics (Time per Million Cells)

This helps compare algorithmic efficiency independent of grid size.

In [ ]:
# Normalized metrics
normalized = df_ok.groupby(['grid_size_label', 'algorithm']).agg(
    time_per_M_cells=('time_per_million_cells_ms', 'mean'),
    nodes_per_cell=('nodes_per_cell', 'mean'),
).round(4)

print('=== Normalized Metrics ===')
print(normalized.to_string())
normalized.to_csv(f'{RESULTS_DIR}/normalized_metrics.csv')

In [ ]:
# Visualization: Normalized comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Time per million cells
ax = axes[0]
for algo in df_ok['algorithm'].unique():
    data = df_ok[df_ok['algorithm'] == algo]
    means = data.groupby('grid_size_label')['time_per_million_cells_ms'].mean()
    means = means.reindex(grid_order)
    ax.plot(range(len(grid_order)), means.values,
            marker='o', linewidth=2, label=algo)
ax.set_xticks(range(len(grid_order)))
ax.set_xticklabels(grid_order)
ax.set_xlabel('Grid Size')
ax.set_ylabel('Time per Million Cells (ms)')
ax.set_title('Normalized Time Efficiency')
ax.legend()

# Nodes per cell
ax = axes[1]
for algo in df_ok['algorithm'].unique():
    data = df_ok[df_ok['algorithm'] == algo]
    means = data.groupby('grid_size_label')['nodes_per_cell'].mean()
    means = means.reindex(grid_order)
    ax.plot(range(len(grid_order)), means.values,
            marker='o', linewidth=2, label=algo)
ax.set_xticks(range(len(grid_order)))
ax.set_xticklabels(grid_order)
ax.set_xlabel('Grid Size')
ax.set_ylabel('Nodes per Cell')
ax.set_title('Search Intensity (Nodes/Cell)')
ax.legend()

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/normalized_metrics.png', bbox_inches='tight')
plt.show()

## 11. Generate Paper-Ready Summary

This generates a structured markdown report you can share for paper updates.

In [ ]:
# Generate and save the paper summary
summary_path = benchmark.save_paper_summary()
analysis_path = benchmark.save_analysis_json()

print(f'Paper summary saved: {summary_path}')
print(f'Analysis JSON saved: {analysis_path}')
print()
print('=' * 70)
print('PAPER SUMMARY (share this for paper updates)')
print('=' * 70)
print(benchmark.generate_paper_summary())

## 12. Export All Results

Save all DataFrames and figures for future reference.

In [ ]:
# Save all DataFrames
df_ok.to_csv(f'{RESULTS_DIR}/all_successful_results.csv', index=False)
df.to_csv(f'{RESULTS_DIR}/all_results_with_failures.csv', index=False)

print('All results exported.')
print(f'\nFiles in {RESULTS_DIR}/:')
for f in sorted(os.listdir(RESULTS_DIR)):
    size = os.path.getsize(os.path.join(RESULTS_DIR, f))
    print(f'  {f:50s} ({size/1024:.1f} KB)')

---

## How to Share Results for Paper Updates

After running the experiments, share the results by:

1. **Copy the paper summary** printed in Section 11 above
2. **Or** provide the content of the generated markdown file:
   ```
   results/large_scale_grids/paper_summary_<timestamp>.md
   ```
3. **For detailed analysis**, share the CSV files:
   - `performance_summary.csv` - Core performance metrics
   - `speedup_analysis.csv` - Speedup factors
   - `scaling_analysis.csv` - Power-law exponents
   - `corridor_analysis.csv` - AILS corridor behavior
   - `statistical_tests.csv` - Significance tests

These results will be used to add a new **Large-Scale Evaluation** section to the paper.